In [1]:
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord
import astropy.units as u

In [31]:
result_table = Simbad.query_object('WASP36')
result_table

main_id,ra,dec,coo_err_maj,coo_err_min,coo_err_angle,coo_wavelength,coo_bibcode,matched_id
,deg,deg,mas,mas,deg,,,
object,float64,float64,float32,float32,int16,str1,object,object
WASP-36,131.58040726648,-8.026948817610002,0.0095,0.0064,90,O,2020yCat.1350....0G,WASP-36


In [38]:
degcoord = SkyCoord(result_table['ra'][0], result_table['dec'][0], unit=(u.deg, u.deg))
print(degcoord.ra.to_string(unit=u.hour, precision=2, sep=':',pad=True))
print(degcoord.dec.to_string(unit=u.deg, sep=':'))
print(degcoord.dec.to_string(unit=u.degree, sep=':', precision=1, pad=True, alwayssign=True))

08:46:19.30
-8:01:37.0157434
-08:01:37.0


In [ ]:
def query_object_coordinates(object_name):
    """
    Query SIMBAD database for object coordinates with user interaction.
    
    This function attempts to find coordinates for an astronomical object by:
    1. First querying SIMBAD with the object name from the FITS header
    2. Showing suggested coordinates and asking for user approval
    3. Allowing the user to input a custom target name for search
    4. Allowing manual input of sexagesimal coordinates if no results found
    
    Args:
        object_name (str): The object name from the FITS header
    
    Returns:
        tuple: (RA, DEC) as strings in the format returned by SIMBAD (e.g., 'HH MM SS.SS', '+DD MM SS.S')
               Returns (None, None) if user cancels or no coordinates found
    """
    print(f"\n{'='*60}")
    print(f"Missing RA/DEC coordinates for object: {object_name}")
    print(f"{'='*60}")
    
    # Try to query SIMBAD with the original object name
    print(f"\nQuerying SIMBAD database for '{object_name}'...")
    
    try:
        result_table = Simbad.query_object(object_name)
        
        if result_table is not None and len(result_table) > 0:
            ra = result_table['RA'][0]
            dec = result_table['DEC'][0]
            
            print(f"\n✓ Found coordinates in SIMBAD:")
            print(f"  RA:  {ra}")
            print(f"  DEC: {dec}")
            
            while True:
                response = input("\nUse these coordinates? (yes/no): ").strip().lower()
                if response in ['yes', 'y']:
                    print("✓ Using SIMBAD coordinates.")
                    return ra, dec
                elif response in ['no', 'n']:
                    break
                else:
                    print("Please enter 'yes' or 'no'.")
        else:
            print(f"✗ No results found for '{object_name}' in SIMBAD.")
    
    except Exception as e:
        print(f"✗ Error querying SIMBAD: {e}")
    
    # Option to search with a different name
    while True:
        print("\nOptions:")
        print("  1. Search with a different target name")
        print("  2. Manually input coordinates")
        print("  3. Skip (leave coordinates empty)")
        
        choice = input("\nEnter your choice (1/2/3): ").strip()
        
        if choice == '1':
            custom_name = input("\nEnter target name to search: ").strip()
            if not custom_name:
                print("No name entered. Please try again.")
                continue
            
            print(f"Querying SIMBAD for '{custom_name}'...")
            try:
                result_table = Simbad.query_object(custom_name)
                
                if result_table is not None and len(result_table) > 0:
                    ra = result_table['RA'][0]
                    dec = result_table['DEC'][0]
                    
                    print(f"\n✓ Found coordinates:")
                    print(f"  RA:  {ra}")
                    print(f"  DEC: {dec}")
                    
                    while True:
                        response = input("\nUse these coordinates? (yes/no): ").strip().lower()
                        if response in ['yes', 'y']:
                            print("✓ Using SIMBAD coordinates.")
                            return ra, dec
                        elif response in ['no', 'n']:
                            break
                        else:
                            print("Please enter 'yes' or 'no'.")
                else:
                    print(f"✗ No results found for '{custom_name}'.")
            
            except Exception as e:
                print(f"✗ Error querying SIMBAD: {e}")
        
        elif choice == '2':
            print("\nEnter coordinates in sexagesimal format:")
            print("Examples:")
            print("  RA:  12 34 56.78 or 12:34:56.78 or 12h34m56.78s")
            print("  DEC: +12 34 56.7 or +12:34:56.7 or +12d34m56.7s")
            
            ra_input = input("\nEnter RA: ").strip()
            dec_input = input("Enter DEC: ").strip()
            
            if not ra_input or not dec_input:
                print("✗ Both RA and DEC are required.")
                continue
            
            try:
                # Validate the coordinates by parsing them
                coord = SkyCoord(ra_input, dec_input, unit=(u.hourangle, u.deg))
                
                # Convert to the format expected (sexagesimal strings)
                ra_str = coord.ra.to_string(unit=u.hour, sep=' ', precision=2, pad=True)
                dec_str = coord.dec.to_string(unit=u.degree, sep=' ', precision=1, pad=True, alwayssign=True)
                
                print(f"\n✓ Parsed coordinates:")
                print(f"  RA:  {ra_str}")
                print(f"  DEC: {dec_str}")
                
                while True:
                    response = input("\nUse these coordinates? (yes/no): ").strip().lower()
                    if response in ['yes', 'y']:
                        print("✓ Using manually entered coordinates.")
                        return ra_str, dec_str
                    elif response in ['no', 'n']:
                        break
                    else:
                        print("Please enter 'yes' or 'no'.")
            
            except Exception as e:
                print(f"✗ Error parsing coordinates: {e}")
                print("Please check the format and try again.")
        
        elif choice == '3':
            print("\n⚠ Skipping coordinate entry. RA and DEC will be set to 'N/A'.")
            return 'N/A', 'N/A'
        
        else:
            print("Invalid choice. Please enter 1, 2, or 3.")